In [ ]:
import time
notebook_start = time.perf_counter()

import os, json, pandas as pd, numpy as np, joblib
from thermoift import print_model_metrics
from thermoift.rng_utils import get_rng
from sklearn.model_selection import train_test_split, cross_validate

In [ ]:
PLOT_FOLDER = "RF_TabPFN_interfacial_thickness_OUTPUTS"
target      = "interfacial_thickness"
SEED        = 655552
TEST_ROWS   = None
N_ESTIMATORS = 10
MAX_DEPTH    = 3
MAX_DEPTH_CHECKPOINTS = [1, 2, 3, 4, 5, 6, 8, 10, None]
N_ESTIMATOR_CHECKPOINTS = [10, 25, 50, 75, 100, 150, 200, 300, 400, 500]

In [ ]:
import warnings
warnings.filterwarnings(
    "ignore",
    message="TabPFN fit/predict failed at leaf",
    category=UserWarning,
    module="tabpfn_extensions",
)

try:
    import tabpfn
    import tabpfn_extensions
    from tabpfn import TabPFNRegressor
    from tabpfn.constants import ModelVersion
    from tabpfn_extensions.rf_pfn import RandomForestTabPFNRegressor

    os.environ["TABPFN_ALLOW_CPU_LARGE_DATASET"] = "1"

    print(f"TabPFN version: {tabpfn.__version__}")
    print("Selected model version:", ModelVersion.V2)
    print("Using RandomForestTabPFNRegressor")

except ImportError as exc:
    raise ImportError("tabpfn and tabpfn_extensions must be installed in this Python environment.") from exc

n_cpus = int(os.environ.get("SLURM_CPUS_PER_TASK", os.cpu_count() or 4))
os.environ["OMP_NUM_THREADS"]      = str(n_cpus)
os.environ["MKL_NUM_THREADS"]      = str(n_cpus)
os.environ["OPENBLAS_NUM_THREADS"] = str(n_cpus)
os.environ["NUMEXPR_NUM_THREADS"]  = str(n_cpus)
try:
    import torch
    torch.set_num_threads(n_cpus)
except ImportError:
    pass
print(f"Thread limit set to {n_cpus} (SLURM_CPUS_PER_TASK)")


In [ ]:
df = pd.read_csv("../interfacial_results_dataset_A4.csv")
print(f"Number of rows: {len(df)}")

if TEST_ROWS is not None:
    df = df.iloc[:TEST_ROWS].copy()
    print(f"Test mode: using first {TEST_ROWS} rows only")
else:
    print("Full mode: using all rows")

print(f"Total samples: {len(df)}")
print(f"\n{target} statistics:")
print(df[target].describe())

In [ ]:
rng       = get_rng(seed=SEED)

z_columns  = [col for col in df.columns if col.startswith("z_")]
Z_non_zero = [col for col in z_columns if (df[col] != 0).any()]
features   = ["temperature", "pressure"] + Z_non_zero

print(f"Selected features: {features}")

X = df[features]
y = df[target]

# 70/15/15 split
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, random_state=SEED)
X_test,  X_val,  y_test,  y_val  = train_test_split(X_temp, y_temp, test_size=0.50, random_state=SEED)

print(f"\nTraining samples:   {X_train.shape[0]}")
print(f"Testing samples:    {X_test.shape[0]}")
print(f"Validation samples: {X_val.shape[0]}")

In [ ]:
# Train RF + TabPFN model
tabpfn_base = TabPFNRegressor(
    random_state=SEED,
    ignore_pretraining_limits=True,
    fit_mode="fit_preprocessors",
)
rf_tabpfn_model = RandomForestTabPFNRegressor(
    tabpfn=tabpfn_base,
    n_estimators=N_ESTIMATORS,
    max_depth=MAX_DEPTH,
)
rf_tabpfn_model.fit(X_train, y_train)

y_train_pred = rf_tabpfn_model.predict(X_train)
y_test_pred  = rf_tabpfn_model.predict(X_test)
y_val_pred   = rf_tabpfn_model.predict(X_val)

metrics = print_model_metrics(y_train, y_train_pred, y_test, y_test_pred, target, unit="nm", y_val=y_val, y_val_pred=y_val_pred)


In [ ]:
results_df = pd.DataFrame({
    "idx":       np.concatenate([y_train.index, y_test.index, y_val.index]),
    "actual":    np.concatenate([y_train.values, y_test.values, y_val.values]),
    "predicted": np.concatenate([y_train_pred,  y_test_pred,  y_val_pred]),
    "split":     ["train"]*len(y_train) + ["test"]*len(y_test) + ["val"]*len(y_val),
})
os.makedirs(PLOT_FOLDER, exist_ok=True)
results_df.to_csv(os.path.join(PLOT_FOLDER, f"RF_TabPFN_{target}_predictions.csv"), index=False)
print(f"Predictions saved: {len(results_df)} rows")

model_path = os.path.join(PLOT_FOLDER, f"RF_TabPFN_{target}_model.joblib")
joblib.dump(rf_tabpfn_model, model_path)
print(f"Model saved to: {model_path}")

In [ ]:
# n_estimators convergence: RMSE vs number of trees
nestimator_rows = []

for n_estimators in N_ESTIMATOR_CHECKPOINTS:
    nest_tabpfn_base = TabPFNRegressor(
        random_state=SEED,
        ignore_pretraining_limits=True,
        fit_mode="fit_preprocessors",
    )
    nest_model = RandomForestTabPFNRegressor(
        tabpfn=nest_tabpfn_base,
        n_estimators=n_estimators,
        max_depth=MAX_DEPTH,
    )
    nest_model.fit(X_train, y_train)

    train_pred = nest_model.predict(X_train)
    test_pred  = nest_model.predict(X_test)

    train_rmse = float(np.sqrt(np.mean((np.asarray(y_train) - np.asarray(train_pred)) ** 2)))
    test_rmse  = float(np.sqrt(np.mean((np.asarray(y_test)  - np.asarray(test_pred))  ** 2)))
    cv_results = cross_validate(
        nest_model, X_train, y_train, cv=5,
        scoring={"rmse": "neg_root_mean_squared_error"},
        n_jobs=n_cpus,
    )
    cv_rmse = float(-cv_results["test_rmse"].mean())

    nestimator_rows.append({
        "n_estimators": int(n_estimators),
        "train_rmse": train_rmse,
        "test_rmse": test_rmse,
        "cv_rmse": cv_rmse,
    })
    print(f"n_estimators={n_estimators}: train RMSE={train_rmse:.6f}, CV RMSE={cv_rmse:.6f}, test RMSE={test_rmse:.6f}")

nestimator_conv_df = pd.DataFrame(nestimator_rows)
os.makedirs(PLOT_FOLDER, exist_ok=True)
nestimator_conv_path = os.path.join(PLOT_FOLDER, f"RF_TabPFN_{target}_nestimators_convergence.csv")
nestimator_conv_df.to_csv(nestimator_conv_path, index=False)
print(f"Saved: {nestimator_conv_path}")


In [ ]:
# max_depth convergence: RMSE vs tree depth
max_depth_rows = []

for depth in MAX_DEPTH_CHECKPOINTS:
    depth_tabpfn_base = TabPFNRegressor(
        random_state=SEED,
        ignore_pretraining_limits=True,
        fit_mode="fit_preprocessors",
    )
    depth_model = RandomForestTabPFNRegressor(
        tabpfn=depth_tabpfn_base,
        n_estimators=N_ESTIMATORS,
        max_depth=depth,
    )
    depth_model.fit(X_train, y_train)

    train_pred = depth_model.predict(X_train)
    test_pred  = depth_model.predict(X_test)

    train_rmse = float(np.sqrt(np.mean((np.asarray(y_train) - np.asarray(train_pred)) ** 2)))
    test_rmse  = float(np.sqrt(np.mean((np.asarray(y_test)  - np.asarray(test_pred))  ** 2)))
    cv_results = cross_validate(
        depth_model, X_train, y_train, cv=5,
        scoring={"rmse": "neg_root_mean_squared_error"},
        n_jobs=n_cpus,
    )
    cv_rmse = float(-cv_results["test_rmse"].mean())

    max_depth_rows.append({
        "max_depth": "None" if depth is None else str(depth),
        "train_rmse": train_rmse,
        "test_rmse": test_rmse,
        "cv_rmse": cv_rmse,
    })
    print(f"max_depth={depth}: train RMSE={train_rmse:.6f}, CV RMSE={cv_rmse:.6f}, test RMSE={test_rmse:.6f}")

max_depth_conv_df = pd.DataFrame(max_depth_rows)
os.makedirs(PLOT_FOLDER, exist_ok=True)
max_depth_conv_path = os.path.join(PLOT_FOLDER, f"RF_TabPFN_{target}_max_depth_convergence.csv")
max_depth_conv_df.to_csv(max_depth_conv_path, index=False)
print(f"Saved: {max_depth_conv_path}")


In [ ]:
cv_results = cross_validate(
    rf_tabpfn_model, X, y, cv=5,
    scoring={
        "r2":   "r2",
        "rmse": "neg_root_mean_squared_error",
        "mae":  "neg_mean_absolute_error",
    },
    n_jobs=n_cpus,
)

cv_r2_scores   = cv_results["test_r2"]
cv_rmse_scores = -cv_results["test_rmse"]
cv_mae_scores  = -cv_results["test_mae"]

print(f"Cross-Validation R² Scores:   {cv_r2_scores}")
print(f"Mean CV R²:   {cv_r2_scores.mean():.6f} (+/- {cv_r2_scores.std() * 2:.6f})")
print(f"\nCross-Validation RMSE Scores: {cv_rmse_scores}")
print(f"Mean CV RMSE: {cv_rmse_scores.mean():.6f} (+/- {cv_rmse_scores.std() * 2:.6f})")
print(f"\nCross-Validation MAE Scores:  {cv_mae_scores}")
print(f"Mean CV MAE:  {cv_mae_scores.mean():.6f} (+/- {cv_mae_scores.std() * 2:.6f})")

In [ ]:
metrics["cv_r2_scores"]   = cv_r2_scores.tolist()
metrics["cv_r2_mean"]     = float(cv_r2_scores.mean())
metrics["cv_r2_std"]      = float(cv_r2_scores.std())
metrics["cv_rmse_scores"] = cv_rmse_scores.tolist()
metrics["cv_rmse_mean"]   = float(cv_rmse_scores.mean())
metrics["cv_rmse_std"]    = float(cv_rmse_scores.std())
metrics["cv_mae_scores"]  = cv_mae_scores.tolist()
metrics["cv_mae_mean"]    = float(cv_mae_scores.mean())
metrics["cv_mae_std"]     = float(cv_mae_scores.std())
metrics["model"]          = "RF_TabPFN"
metrics["features"]       = features
metrics["target"]         = target
metrics["seed"]           = SEED
metrics["n_estimators"]   = N_ESTIMATORS
metrics["max_depth"]      = MAX_DEPTH

os.makedirs(PLOT_FOLDER, exist_ok=True)
metrics_path = os.path.join(PLOT_FOLDER, f"RF_TabPFN_{target}_metrics.json")
with open(metrics_path, "w") as f:
    json.dump(metrics, f, indent=2, default=str)
print(f"\nMetrics saved to: {metrics_path}")

In [ ]:
notebook_end = time.perf_counter()
elapsed_minutes = (notebook_end - notebook_start) / 60
print(f"Total notebook runtime: {elapsed_minutes:.2f} minutes")